# **01. Data Understanding**

This notebook performs the initial understanding of the Telco Customer Churn dataset.

The goal is to inspect the structure of the data, identify potential data quality issues, analyze the target variable, and prepare a first cleaned version of the dataset for the next stages of the project.

This notebook does not include advanced exploratory analysis or model training. Those steps will be developed in later notebooks.

## 1. Introduction

Customer churn analysis aims to identify customers who are likely to leave a company. In this project, the objective is to analyze churn patterns and later build a predictive model that estimates the probability of customer churn.

Before performing exploratory data analysis or machine learning, it is necessary to understand the dataset and detect possible data quality issues.

## 2. Data Loading and Initial Overview

This section loads the dataset and provides a first overview of its structure, including the number of rows, columns, available variables, and sample records.

The dataset is loaded from the `data/raw/` directory to keep the original data separated from processed versions.

In [15]:
import pandas as pd
import numpy as np
from pathlib import Path

In [16]:
DATA_PATH = Path("../data/raw/telco-customer-churn.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [17]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 7043
Number of columns: 21


The dataset contains 7,043 customer records and 21 columns. Each row represents a unique customer, while the columns describe demographics, subscribed services, contract information, billing details, charges, and churn status.

### 2.1 Column Overview

The dataset includes the following types of variables:

- Customer identifier: `customerID`
- Demographic variables: `gender`, `SeniorCitizen`, `Partner`, `Dependents`
- Service-related variables: `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
- Contract and billing variables: `Contract`, `PaperlessBilling`, `PaymentMethod`
- Charge variables: `tenure`, `MonthlyCharges`, `TotalCharges`
- Target variable: `Churn`

In [18]:
df.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

## 3. Data Quality Assessment

This section checks data types, missing values, duplicated records, and hidden data quality issues.

### 3.1 Data Types

Checking data types helps identify variables that may require conversion before analysis or model training.

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [20]:
df.dtypes.sort_values()

SeniorCitizen         int64
tenure                int64
MonthlyCharges      float64
customerID           object
PaymentMethod        object
PaperlessBilling     object
Contract             object
StreamingMovies      object
StreamingTV          object
TechSupport          object
DeviceProtection     object
OnlineBackup         object
OnlineSecurity       object
InternetService      object
MultipleLines        object
PhoneService         object
Dependents           object
Partner              object
gender               object
TotalCharges         object
Churn                object
dtype: object

Most variables are categorical and are loaded as `object`. The numerical variables automatically detected are `SeniorCitizen`, `tenure`, and `MonthlyCharges`.

However, `TotalCharges` appears as an object column, even though it represents a monetary value. This suggests that the column may contain non-numeric or blank values.

### 3.2 Missing Values

The dataset is checked for explicit missing values.

In [21]:
missing_summary = pd.DataFrame({
    "missing_values": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
}).sort_values(by="missing_values", ascending=False)

missing_summary

,missing_values,missing_percentage
customerID,0,0.0
gender,0,0.0
SeniorCitizen,0,0.0
Partner,0,0.0
Dependents,0,0.0
tenure,0,0.0
PhoneService,0,0.0
MultipleLines,0,0.0
InternetService,0,0.0
OnlineSecurity,0,0.0


No explicit missing values are detected using `isnull()`. However, missing values may still be encoded as blank spaces or empty strings, so this needs to be checked separately.

### 3.3 Duplicated Records

Duplicated rows and duplicated customer identifiers are checked because they could bias the analysis.

In [22]:
duplicated_rows = df.duplicated().sum()
duplicated_customer_ids = df["customerID"].duplicated().sum()

print(f"Duplicated rows: {duplicated_rows}")
print(f"Duplicated customer IDs: {duplicated_customer_ids}")

Duplicated rows: 0
Duplicated customer IDs: 0


No duplicated rows or duplicated customer identifiers are found. Therefore, each row can be treated as a unique customer record.

### 3.4 Hidden Missing Values in `TotalCharges`

Since `TotalCharges` was loaded as an object variable, it is necessary to check whether it contains blank spaces or non-numeric values.

In [23]:
blank_total_charges = (df["TotalCharges"].astype(str).str.strip() == "").sum()

print(f"Blank values in TotalCharges: {blank_total_charges}")

Blank values in TotalCharges: 11


There are 11 blank values in `TotalCharges`. These records have `tenure` equal to 0, which suggests they correspond to new customers who have not yet accumulated total charges.

This explains why `TotalCharges` was loaded as an object instead of as a numerical variable.

In [24]:
df[df["TotalCharges"].astype(str).str.strip() == ""]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,,No


## 4. Target and Feature Understanding

This section provides an initial understanding of the target variable and the main feature groups.

### 4.1 Target Distribution

The target variable is `Churn`, which indicates whether a customer left the company.

In [25]:
target_summary = pd.DataFrame({
    "count": df["Churn"].value_counts(),
    "percentage": (df["Churn"].value_counts(normalize=True) * 100).round(2)
})

target_summary

,count,percentage
Churn,,
No,5174,73.46
Yes,1869,26.54


The target variable is moderately imbalanced. Most customers did not churn, while approximately one quarter of customers did.

This imbalance should be considered during model evaluation. Accuracy alone may not be sufficient, so later stages should also use metrics such as precision, recall, F1-score, ROC-AUC, and confusion matrix.

In [26]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


At this stage, `TotalCharges` does not appear in the numerical summary because it was loaded as an object. This issue will be corrected in the initial cleaning section.

### 4.3 Categorical Variables

Categorical variables are inspected by checking their number of unique values and sample categories.

In [27]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

categorical_summary = pd.DataFrame({
    "column": categorical_columns,
    "unique_values": [df[col].nunique() for col in categorical_columns],
    "sample_values": [df[col].unique()[:5] for col in categorical_columns]
})

categorical_summary

,column,unique_values,sample_values
0,customerID,7043,"[7590-VHVEG, 5575-GNVDE, 3668-QPYBK, 7795-CFOC..."
1,gender,2,"[Female, Male]"
2,Partner,2,"[Yes, No]"
3,Dependents,2,"[No, Yes]"
4,PhoneService,2,"[No, Yes]"
5,MultipleLines,3,"[No phone service, No, Yes]"
6,InternetService,3,"[DSL, Fiber optic, No]"
7,OnlineSecurity,3,"[No, Yes, No internet service]"
8,OnlineBackup,3,"[Yes, No, No internet service]"
9,DeviceProtection,3,"[No, Yes, No internet service]"


Most categorical variables have a limited number of unique values, which makes them suitable for encoding in later machine learning stages.

Some service-related columns contain categories such as `No internet service` or `No phone service`. These values are valid categories and should not be treated as missing values.

### 4.4 Feature Classification

The variables are grouped according to their role in the project.

In [29]:
id_column = "customerID"
target_column = "Churn"

numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

binary_categorical_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

multi_class_categorical_features = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

print(f"ID column: {id_column}")
print(f"Target column: {target_column}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Binary categorical features: {len(binary_categorical_features)}")
print(f"Multi-class categorical features: {len(multi_class_categorical_features)}")

ID column: customerID
Target column: Churn
Numerical features: 3
Binary categorical features: 6
Multi-class categorical features: 10


The `customerID` column is an identifier and should not be used as a predictive feature.

For future model training, categorical variables will need to be encoded, the target variable will need to be converted to binary format, and numerical variables may need scaling depending on the selected model.

## 5. Initial Cleaning and Conclusions

This section applies a minimal initial correction to the dataset and summarizes the main findings from the data understanding process.

### 5.1 Convert `TotalCharges` to Numeric

The `TotalCharges` column is converted from object to numeric. Blank spaces are converted into missing values using `errors="coerce"`.

In [30]:
df_clean = df.copy()

df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

df_clean["TotalCharges"].dtype

dtype('float64')

In [32]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.00,0.000,0.0000,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.00,29.000,55.0000,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.50,70.350,89.8500,118.75
TotalCharges,7032.0,2283.300441,2266.771362,18.80,401.45,1397.475,3794.7375,8684.80


After conversion, `TotalCharges` is correctly represented as a numerical variable. The 11 blank values are now represented as missing values.

Since these records correspond to customers with `tenure = 0`, a reasonable strategy in the preprocessing stage may be to replace these missing values with 0. However, this decision will be handled explicitly in the preprocessing workflow.

### 5.2 Basic Sanity Checks

Simple checks are performed to detect impossible or suspicious numerical values.

In [33]:
numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in numeric_columns:
    print(f"{col}")
    print(f"Minimum: {df_clean[col].min()}")
    print(f"Maximum: {df_clean[col].max()}")
    print()

tenure
Minimum: 0
Maximum: 72

MonthlyCharges
Minimum: 18.25
Maximum: 118.75

TotalCharges
Minimum: 18.8
Maximum: 8684.8



In [34]:
print("Customers with tenure equal to 0:")
print((df_clean["tenure"] == 0).sum())

print("\nCustomers with MonthlyCharges less than or equal to 0:")
print((df_clean["MonthlyCharges"] <= 0).sum())

print("\nCustomers with TotalCharges less than 0:")
print((df_clean["TotalCharges"] < 0).sum())

Customers with tenure equal to 0:
11

Customers with MonthlyCharges less than or equal to 0:
0

Customers with TotalCharges less than 0:
0


No negative charge values are detected. The records with `tenure = 0` correspond to the customers whose `TotalCharges` value was originally blank.

### 5.3 Save Initial Cleaned Dataset

The corrected version of the dataset is saved in the `data/processed/` directory.

At this stage, only the data type correction for `TotalCharges` has been applied.

In [35]:
OUTPUT_PATH = Path("../data/processed/telco_customer_churn_initial_clean.csv")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"File saved at: {OUTPUT_PATH}")

File saved at: ..\data\processed\telco_customer_churn_initial_clean.csv


### 5.4 Initial Findings

The initial data understanding process provides the following conclusions:

- The dataset contains 7,043 customers and 21 columns.
- Each row represents a unique customer.
- No duplicated rows or duplicated customer identifiers were found.
- No explicit missing values were initially detected.
- The target variable is moderately imbalanced, with approximately 26.5% churned customers.
- `TotalCharges` was incorrectly loaded as an object column.
- `TotalCharges` contained 11 blank values.
- These blank values correspond to customers with `tenure = 0`.
- After conversion, `TotalCharges` contains 11 missing values.
- The dataset is suitable for exploratory analysis after handling the `TotalCharges` issue.

### 5.5 Next Steps

The next notebook will focus on exploratory data analysis.

The main objectives will be:

- Analyze churn rate by demographic variables.
- Analyze churn rate by contract type and payment method.
- Study the relationship between tenure, monthly charges, total charges, and churn.
- Identify customer segments with higher churn risk.
- Generate business-oriented insights for customer retention.